# Feature Engineering

**Objective:** Build a comprehensive feature set from the merged hourly dataset for ML model training.

**Features created:**
- **Temporal:** hour, day_of_week, month, day_of_month, is_weekend
- **Cyclical:** hour_sin, hour_cos, dow_sin, dow_cos (captures circular time patterns)
- **Lag:** occupancy at 1h, 3h, 6h, 12h, 24h, 168h (1 week); volume at 24h
- **Rolling:** mean over 6h/12h/24h windows, 24h std deviation, 1h difference
- **Spatial:** charge_density (piles/km²), total_price (electricity + service fee)

**Input:** `data/processed/merged_hourly_data_01.csv & merged_hourly_data_02.csv`  
**Output:** `data/processed/final_featured_dataset.csv`

## Import Required Libraries


In [1]:
import pandas as pd
import numpy as np
import time as timer
from pathlib import Path

print("Libraries loaded!")

Libraries loaded!


## 1. Load Merged Dataset

In [2]:
DATA_PATH = Path("../data/processed")

# Load both parts
df1 = pd.read_csv(DATA_PATH / "merged_hourly_data_01.csv")
df2 = pd.read_csv(DATA_PATH / "merged_hourly_data_02.csv")

# Concatenate vertically (stack rows)
df = pd.concat([df1, df2], ignore_index=True)

# Convert time column
df['time'] = pd.to_datetime(df['time'])

df.head()

,time,zone_id,occupancy,volume,duration,e_price,s_price,longitude,latitude,charge_count,area,perimeter,num_stations,total_piles,mean_station_lat,mean_station_lon
0,2022-09-01 00:00:00,102,17.0,56.291668,8.833333,0.924,0.856,114.102962,22.540413,30,714419.3333,3501.2154,4,30,22.538339,114.102945
1,2022-09-01 01:00:00,102,16.0,50.458332,8.166667,0.924,0.856,114.102962,22.540413,30,714419.3333,3501.2154,4,30,22.538339,114.102945
2,2022-09-01 02:00:00,102,16.0,55.708332,8.916667,0.924,0.856,114.102962,22.540413,30,714419.3333,3501.2154,4,30,22.538339,114.102945
3,2022-09-01 03:00:00,102,16.0,57.750000,9.166667,0.924,0.856,114.102962,22.540413,30,714419.3333,3501.2154,4,30,22.538339,114.102945
4,2022-09-01 04:00:00,102,17.0,58.333332,9.250000,0.924,0.856,114.102962,22.540413,30,714419.3333,3501.2154,4,30,22.538339,114.102945


## 2. Temporal Features

Extract time components from the timestamp. These capture daily rhythm (hour), weekly patterns (day_of_week), and seasonal trends (month).

In [3]:
df['hour'] = df['time'].dt.hour.astype(np.int8)
df['day_of_week'] = df['time'].dt.dayofweek.astype(np.int8)
df['month'] = df['time'].dt.month.astype(np.int8)
df['day_of_month'] = df['time'].dt.day.astype(np.int8)
df['is_weekend'] = (df['day_of_week'] >= 5).astype(np.int8)

print("Temporal features created: hour, day_of_week, month, day_of_month, is_weekend")

Temporal features created: hour, day_of_week, month, day_of_month, is_weekend


## 3. Cyclical Encoding

Hour 23 and hour 0 are adjacent in reality but appear far apart (23 vs 0) to tree models.  
Sin/cos encoding maps them onto a circle so the model sees their proximity.

Same logic for day_of_week: Sunday (6) is adjacent to Monday (0).

In [4]:
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24).astype(np.float32)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24).astype(np.float32)
df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7).astype(np.float32)
df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7).astype(np.float32)

print("Cyclical features created: hour_sin, hour_cos, dow_sin, dow_cos")

Cyclical features created: hour_sin, hour_cos, dow_sin, dow_cos


## 4. Spatial Features

- **charge_density:** Charging piles per km² — normalizes infrastructure by zone area
- **total_price:** Combined electricity + service price — article shows price combinations improve predictions (Table 5)

In [5]:
# Charge density: piles per km² (area is in m², convert to km²)
df['charge_density'] = (df['charge_count'] / (df['area'] / 1e6)).astype(np.float32)

# Total price: electricity + service fee
df['total_price'] = (df['e_price'] + df['s_price']).astype(np.float32)

print("Spatial features created: charge_density, total_price")
print(f"charge_density range: {df['charge_density'].min():.1f} - {df['charge_density'].max():.1f} piles/km²")
print(f"total_price range: {df['total_price'].min():.2f} - {df['total_price'].max():.2f} CNY/kWh")

Spatial features created: charge_density, total_price
charge_density range: 0.7 - 378.8 piles/km²
total_price range: 0.88 - 2.26 CNY/kWh


## 5. Lag Features

Lag features capture historical demand patterns:
- **Short-term:** 1h, 3h, 6h ago — recent momentum
- **Medium-term:** 12h, 24h ago — same time yesterday, half-day pattern
- **Long-term:** 168h (1 week) ago — same hour, same weekday last week
- **Cross-variable:** volume 24h ago — yesterday's energy usage may predict today's occupancy

All lags are computed **per zone** (grouped by zone_id) to avoid data leakage between zones.

In [6]:
# Sort by zone and time (required for correct lag/shift operations)
df = df.sort_values(['zone_id', 'time']).reset_index(drop=True)

# Occupancy lags
g_occ = df.groupby('zone_id', sort=False)['occupancy']
for lag in [1, 3, 6, 12, 24, 168]:
    df[f'occ_lag_{lag}h'] = g_occ.shift(lag).astype(np.float32)

# Volume lag (cross-variable)
df['vol_lag_24h'] = df.groupby('zone_id', sort=False)['volume'].shift(24).astype(np.float32)

print("Lag features created:")
print("Occupancy lags: 1h, 3h, 6h, 12h, 24h, 168h")
print("Volume lag: 24h")

Lag features created:
Occupancy lags: 1h, 3h, 6h, 12h, 24h, 168h
Volume lag: 24h


## 6. Rolling Features

Rolling statistics over shifted windows to avoid data leakage (shift by 1 first, then roll).

- **Rolling mean (6h/12h/24h):** Smoothed recent trend
- **Rolling std (24h):** Volatility/variability in last 24 hours
- **Diff (1h):** Hour-over-hour change — captures acceleration/deceleration

All operations are **grouped by zone_id** to prevent cross-zone contamination.

In [7]:
# Shifted occupancy (shift by 1 to prevent leakage)
g_occ = df.groupby('zone_id', sort=False)['occupancy']
shifted = g_occ.shift(1)

#### Rolling Mean

In [8]:
# Rolling mean (last 6 hours)
df['occ_rmean_6h'] = (
        shifted.groupby(df['zone_id'], sort=False)
        .rolling(6, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
        .astype(np.float32)
)

# Rolling mean (last 12 hours)
df['occ_rmean_12h'] = (
        shifted.groupby(df['zone_id'], sort=False)
        .rolling(12, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
        .astype(np.float32)
)

# Rolling mean (last 24 hours)
df['occ_rmean_24h'] = (
        shifted.groupby(df['zone_id'], sort=False)
        .rolling(24, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
        .astype(np.float32)
)

#### Rolling Standard Deviation

In [9]:
# Rolling Standard Deviation (last 24 hours)
df['occ_rstd_24h'] = (
    shifted.groupby(df['zone_id'], sort=False)
    .rolling(24, min_periods=1)
    .std()
    .reset_index(level=0, drop=True)
    .astype(np.float32)
)

#### Hourly Difference

In [10]:
# Hourly difference
df['occ_diff_1h'] = g_occ.diff(1).astype(np.float32)

print("Rolling features created:")
print("Rolling means: 6h, 12h, 24h")
print("Rolling std: 24h")
print("Diff: 1h")

Rolling features created:
Rolling means: 6h, 12h, 24h
Rolling std: 24h
Diff: 1h


## 7. Drop NaN Rows & Final Check

Rows where the value is NaN must be dropped - they don't have enough history for the model to use.

In [11]:
before = len(df)
df = df.dropna().reset_index(drop=True)
print(f"Dropped {before - len(df):,} rows")
print(f"Final shape: {df.shape}")

Dropped 46,200 rows
Final shape: (1148400, 39)


In [12]:
print("Final dataset data types:")
df.dtypes

Final dataset data types:


time                datetime64[ns]
zone_id                      int64
occupancy                  float64
volume                     float64
duration                   float64
e_price                    float64
s_price                    float64
longitude                  float64
latitude                   float64
charge_count                 int64
area                       float64
perimeter                  float64
num_stations                 int64
total_piles                  int64
mean_station_lat           float64
mean_station_lon           float64
hour                          int8
day_of_week                   int8
month                         int8
day_of_month                  int8
is_weekend                    int8
hour_sin                   float32
hour_cos                   float32
dow_sin                    float32
dow_cos                    float32
charge_density             float32
total_price                float32
occ_lag_1h                 float32
occ_lag_3h          

## 8. Save Final Dataset

In [13]:
OUTPUT_DIR = Path("../data/processed")

splits = np.array_split(df, 4)

for i, part in enumerate(splits, start=1):
    output_path = OUTPUT_DIR / f"final_featured_dataset_part_0{i}.csv"
    
    part.to_csv(output_path, index=False)
    
    print(f"\nSaved: {output_path}")
    print(f"Shape: {part.shape}")
    print(f"File size: {output_path.stat().st_size / 1e6:.1f} MB")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)



Saved: ../data/processed/final_featured_dataset_part_01.csv
Shape: (287100, 39)
File size: 80.2 MB

Saved: ../data/processed/final_featured_dataset_part_02.csv
Shape: (287100, 39)
File size: 77.7 MB

Saved: ../data/processed/final_featured_dataset_part_03.csv
Shape: (287100, 39)
File size: 78.5 MB

Saved: ../data/processed/final_featured_dataset_part_04.csv
Shape: (287100, 39)
File size: 79.7 MB
